# Colorado Housing Opportunity Analysis

This project evaluates Colorado ZIP codes using two complementary indicators:

- **Historical momentum:** three-year Zillow Home Value Index (ZHVI) appreciation.
- **Relative affordability:** home value divided by Census ACS median household income.

The final opportunity score weights momentum at 70% and affordability at 30%. Historical tests compare the ZIP codes selected by each strategy with the remaining ZIP codes in the following year. The score is an **exploratory market-screening index**, not a forecast or investment recommendation.


## Reproducibility

Expected repository layout:

```text
data/
  Predictive_Analysis_1.csv
  zillow_colorado_clean.csv
notebooks/
  colorado_housing_opportunity_analysis.ipynb
outputs/
```

Install `pandas`, `numpy`, `requests`, `scipy`, and `statsmodels`. The Census API works without a key for light use. To use a key, set the `CENSUS_API_KEY` environment variable; never place credentials in the notebook.

The analysis uses the 2024 ACS five-year estimate for the September 2026 snapshot because it is the selected available income vintage. Zillow values may be revised after download.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import requests
from scipy.stats import spearmanr
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# These defaults work when the notebook is opened from notebooks/.
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DATA_DIR = Path(os.getenv("PROJECT_DATA_DIR", PROJECT_ROOT / "data"))
OUTPUT_DIR = Path(os.getenv("PROJECT_OUTPUT_DIR", PROJECT_ROOT / "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HISTORICAL_FILE = DATA_DIR / "Predictive_Analysis_1.csv"
ZILLOW_FILE = DATA_DIR / "zillow_colorado_clean.csv"
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

CURRENT_DATE = pd.Timestamp("2026-09-30")
MOMENTUM_START_DATE = pd.Timestamp("2023-09-30")
CURRENT_ACS_YEAR = 2024
TOP_SHARE = 0.20
MOMENTUM_WEIGHT = 0.70
AFFORDABILITY_WEIGHT = 0.30


## 1. Historical modeling data

The historical dataset contains one row per ZIP code and prediction year. `past_3yr_growth` is known at ranking time; `future_1yr_growth` is used only to evaluate subsequent performance.


In [ ]:
required_historical_columns = {
    "zip_code", "year", "home_value", "past_1yr_growth",
    "past_3yr_growth", "past_5yr_growth", "future_1yr_growth",
}

if not HISTORICAL_FILE.exists():
    raise FileNotFoundError(
        f"Missing {HISTORICAL_FILE}. Add the historical modeling CSV to the data directory "
        "or set PROJECT_DATA_DIR."
    )

historical_df = pd.read_csv(HISTORICAL_FILE, dtype={"zip_code": str})
historical_df["zip_code"] = historical_df["zip_code"].str.zfill(5)

missing_columns = required_historical_columns - set(historical_df.columns)
if missing_columns:
    raise ValueError(f"Historical data is missing columns: {sorted(missing_columns)}")
if historical_df.duplicated(["zip_code", "year"]).any():
    raise ValueError("Historical data contains duplicate ZIP-year observations.")

historical_df["market_growth"] = historical_df.groupby("year")[
    "future_1yr_growth"
].transform("mean")
historical_df["relative_growth"] = (
    historical_df["future_1yr_growth"] - historical_df["market_growth"]
)

pd.DataFrame({
    "rows": [len(historical_df)],
    "zip_codes": [historical_df["zip_code"].nunique()],
    "first_year": [historical_df["year"].min()],
    "last_year": [historical_df["year"].max()],
    "duplicate_zip_years": [historical_df.duplicated(["zip_code", "year"]).sum()],
})


## 2. Momentum backtest

For each historical year, the strategy selects the top 20% of eligible ZIP codes. Outperformance is the selected group’s mean subsequent one-year growth minus the mean for all remaining eligible ZIP codes. Group size is calculated dynamically rather than hard-coded.


In [ ]:
def evaluate_top_share(data, score_column, outcome_column="relative_growth", top_share=TOP_SHARE):
    # Compare a high-score selection with the remaining eligible observations by year.
    rows = []
    for year, year_data in data.groupby("year"):
        eligible = year_data.dropna(subset=[score_column, outcome_column]).copy()
        if len(eligible) < 20:
            continue

        n_selected = max(1, int(np.ceil(len(eligible) * top_share)))
        eligible["selected"] = (
            eligible[score_column].rank(ascending=False, method="first") <= n_selected
        )
        selected_growth = eligible.loc[eligible["selected"], outcome_column].mean()
        remaining_growth = eligible.loc[~eligible["selected"], outcome_column].mean()
        rows.append({
            "year": year,
            "eligible_zip_codes": len(eligible),
            "selected_zip_codes": n_selected,
            "outperformance_pp": selected_growth - remaining_growth,
        })
    return pd.DataFrame(rows)


momentum_backtest = evaluate_top_share(historical_df, "past_3yr_growth")
momentum_backtest["strategy"] = "3-year momentum"
momentum_backtest


In [ ]:
momentum_correlations = []
for year, year_data in historical_df.groupby("year"):
    eligible = year_data.dropna(subset=["past_3yr_growth", "relative_growth"])
    correlation, p_value = spearmanr(
        eligible["past_3yr_growth"], eligible["relative_growth"]
    )
    momentum_correlations.append({
        "year": year,
        "spearman_correlation": correlation,
        "p_value": p_value,
    })

pd.DataFrame(momentum_correlations)


## 3. Census income and affordability

Historical prediction year *t* is matched to ACS vintage *t − 1*. Negative Census estimate codes are treated as missing. The API request is reproducible and does not expose a credential.


In [ ]:
def fetch_acs_income(years, api_key=None):
    frames = []
    for acs_year in years:
        url = f"https://api.census.gov/data/{acs_year}/acs/acs5"
        params = {
            "get": "NAME,B19013_001E",
            "for": "zip code tabulation area:*",
        }
        if api_key:
            params["key"] = api_key

        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        yearly = pd.DataFrame(payload[1:], columns=payload[0]).rename(columns={
            "B19013_001E": "median_income",
            "zip code tabulation area": "zip_code",
        })
        yearly["zip_code"] = yearly["zip_code"].astype(str).str.zfill(5)
        yearly["median_income"] = pd.to_numeric(yearly["median_income"], errors="coerce")
        yearly.loc[yearly["median_income"] <= 0, "median_income"] = np.nan
        yearly["acs_year"] = acs_year
        frames.append(yearly[["zip_code", "acs_year", "median_income"]])

    return pd.concat(frames, ignore_index=True)


historical_acs_years = sorted((historical_df["year"] - 1).unique())
income_history = fetch_acs_income(historical_acs_years, CENSUS_API_KEY)
income_history["year"] = income_history["acs_year"] + 1

modeling_df = historical_df.merge(
    income_history[["zip_code", "year", "median_income"]],
    on=["zip_code", "year"],
    how="left",
    validate="one_to_one",
)
modeling_df["price_income_ratio"] = modeling_df["home_value"] / modeling_df["median_income"]

modeling_df.groupby("year").agg(
    zip_codes=("zip_code", "nunique"),
    valid_income=("median_income", "count"),
    median_price_income_ratio=("price_income_ratio", "median"),
)


## 4. Combined momentum and affordability strategy

Both inputs are converted to within-year percentile scores. Higher momentum is better; a lower price-to-income ratio is more affordable. The combined score applies the stated 70/30 weighting.


In [ ]:
modeling_df["momentum_score"] = modeling_df.groupby("year")["past_3yr_growth"].rank(
    pct=True
)
modeling_df["affordability_score"] = modeling_df.groupby("year")["price_income_ratio"].rank(
    ascending=False, pct=True
)
modeling_df["combined_score"] = (
    MOMENTUM_WEIGHT * modeling_df["momentum_score"]
    + AFFORDABILITY_WEIGHT * modeling_df["affordability_score"]
)

combined_backtest = evaluate_top_share(modeling_df, "combined_score")
combined_backtest["strategy"] = "Momentum + affordability"

backtest_results = pd.concat(
    [momentum_backtest, combined_backtest], ignore_index=True
).sort_values(["year", "strategy"])

backtest_results.pivot(
    index="year", columns="strategy", values="outperformance_pp"
).round(3)


In [ ]:
backtest_summary = backtest_results.groupby("strategy", as_index=False).agg(
    average_outperformance_pp=("outperformance_pp", "mean"),
    positive_periods=("outperformance_pp", lambda values: int((values > 0).sum())),
    periods=("outperformance_pp", "count"),
)
backtest_summary.round(3)


In [ ]:
# Diagnostic association model; this does not establish causality.
regression_data = modeling_df.dropna(
    subset=["relative_growth", "momentum_score", "affordability_score", "zip_code"]
).copy()

diagnostic_model = smf.ols(
    "relative_growth ~ momentum_score + affordability_score + C(year)",
    data=regression_data,
).fit(cov_type="cluster", cov_kwds={"groups": regression_data["zip_code"]})

pd.DataFrame({
    "coefficient": diagnostic_model.params,
    "std_error": diagnostic_model.bse,
    "p_value": diagnostic_model.pvalues,
}).loc[["momentum_score", "affordability_score"]].round(3)


## 5. September 2026 opportunity ranking

The current screen compares September 2026 ZHVI with September 2023 ZHVI, then combines three-year appreciation with 2024 ACS affordability. The exact dates and income vintage are parameters near the top of the notebook.


In [ ]:
required_zillow_columns = {"zip_code", "date", "zhvi"}
if not ZILLOW_FILE.exists():
    raise FileNotFoundError(
        f"Missing {ZILLOW_FILE}. Add the cleaned monthly Zillow CSV to the data directory "
        "or set PROJECT_DATA_DIR."
    )

monthly_zillow_df = pd.read_csv(ZILLOW_FILE, dtype={"zip_code": str})
missing_columns = required_zillow_columns - set(monthly_zillow_df.columns)
if missing_columns:
    raise ValueError(f"Zillow data is missing columns: {sorted(missing_columns)}")

monthly_zillow_df["zip_code"] = monthly_zillow_df["zip_code"].str.zfill(5)
monthly_zillow_df["date"] = pd.to_datetime(monthly_zillow_df["date"])

available_dates = set(monthly_zillow_df["date"])
for required_date in [CURRENT_DATE, MOMENTUM_START_DATE]:
    if required_date not in available_dates:
        raise ValueError(
            f"Required Zillow date {required_date.date()} is unavailable. "
            f"Latest available date: {monthly_zillow_df['date'].max().date()}"
        )

current_values = monthly_zillow_df.loc[
    monthly_zillow_df["date"].eq(CURRENT_DATE), ["zip_code", "zhvi"]
].rename(columns={"zhvi": "current_value"})
prior_values = monthly_zillow_df.loc[
    monthly_zillow_df["date"].eq(MOMENTUM_START_DATE), ["zip_code", "zhvi"]
].rename(columns={"zhvi": "value_3yr_ago"})

current_rankings = current_values.merge(
    prior_values, on="zip_code", how="inner", validate="one_to_one"
)
current_rankings["growth_3yr"] = (
    current_rankings["current_value"] / current_rankings["value_3yr_ago"] - 1
) * 100

current_rankings.describe(include="all")


In [ ]:
current_income = fetch_acs_income([CURRENT_ACS_YEAR], CENSUS_API_KEY)
current_rankings = current_rankings.merge(
    current_income[["zip_code", "median_income"]],
    on="zip_code",
    how="left",
    validate="one_to_one",
)
current_rankings["price_income_ratio"] = (
    current_rankings["current_value"] / current_rankings["median_income"]
)

eligible = current_rankings.dropna(
    subset=["growth_3yr", "price_income_ratio"]
).copy()
eligible["momentum_score"] = eligible["growth_3yr"].rank(pct=True) * 100
eligible["affordability_score"] = eligible["price_income_ratio"].rank(
    ascending=False, pct=True
) * 100
eligible["opportunity_score"] = (
    MOMENTUM_WEIGHT * eligible["momentum_score"]
    + AFFORDABILITY_WEIGHT * eligible["affordability_score"]
)
eligible = eligible.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
eligible["overall_rank"] = np.arange(1, len(eligible) + 1)

display_columns = [
    "overall_rank", "zip_code", "current_value", "growth_3yr",
    "median_income", "price_income_ratio", "opportunity_score",
]
eligible[display_columns].head(20).round(2)


In [ ]:
ranking_output = OUTPUT_DIR / "colorado_opportunity_rankings_2026.csv"
backtest_output = OUTPUT_DIR / "backtesting_tableau.csv"

eligible.to_csv(ranking_output, index=False)
backtest_results.to_csv(backtest_output, index=False)

print(f"Saved {len(eligible):,} ranked ZIP codes to {ranking_output}")
print(f"Saved {len(backtest_results):,} backtest rows to {backtest_output}")


## Interpretation and limitations

- The score ranks ZIP codes on two historical indicators; it does not predict individual property returns.
- Historical backtests are retrospective and cover a limited number of market periods.
- The 70/30 weighting is a transparent analytical choice, not an independently validated optimum.
- Zillow estimates can be revised, and results may change when the source data is refreshed.
- ACS five-year estimates have sampling uncertainty and lag the September 2026 Zillow snapshot.
- ZIP-level averages omit property condition, transaction costs, financing, taxes, insurance, rents, liquidity, and neighborhood-level variation.
- Strategy development and evaluation use overlapping historical evidence, so reported performance should be treated as exploratory rather than out-of-sample proof.
